In [ ]:
import os
import pandas as pd

from socceraction.data.statsbomb import StatsBombLoader

from football_ai.data import (
    build_and_save_vaep_for_competition_season,
    resolve_competition_season_ids,
    slug,
)
from football_ai.evaluation import evaluate_binary, get_positive_class_scores

pd.set_option('display.max_rows', 10)

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# All data pipeline functions are imported from football_ai.data:
#   slug, resolve_competition_season_ids, build_and_save_vaep_for_competition_season, etc.
# All evaluation functions are imported from football_ai.evaluation:
#   get_positive_class_scores, evaluate_binary

In [3]:
# Paths
data_root = '../../open-data/data'  # StatsBomb open-data JSON folder
output_dir = os.path.join('..', 'data', 'spadl_data_women_leagues')
os.makedirs(output_dir, exist_ok=True)

# Loader
SBL = StatsBombLoader(getter='local', root=data_root)

print('output_dir:', output_dir)

output_dir: ../data/spadl_data_women_leagues


In [4]:
# List available competitions/seasons
df_competitions = SBL.competitions().copy()
cols = ['competition_id', 'season_id', 'competition_name', 'season_name']
display(df_competitions[cols].sort_values(['competition_name', 'season_name']).reset_index(drop=True))

,competition_id,season_id,competition_name,season_name
0,9,27,1. Bundesliga,2015/2016
1,9,281,1. Bundesliga,2023/2024
2,1267,107,African Cup of Nations,2023
3,16,276,Champions League,1970/1971
4,16,71,Champions League,1971/1972
...,...,...,...,...
70,35,75,UEFA Europa League,1988/1989
71,53,106,UEFA Women's Euro,2022
72,53,315,UEFA Women's Euro,2025
73,72,30,Women's World Cup,2019


In [5]:
for el in df_competitions['competition_name'].value_counts().index:
    print(el)


Champions League
La Liga
FIFA World Cup
Copa del Rey
Ligue 1
FA Women's Super League
Women's World Cup
1. Bundesliga
UEFA Euro
Liga Profesional
UEFA Women's Euro
Premier League
Serie A
Copa America
African Cup of Nations
FIFA U20 World Cup
Indian Super league
Major League Soccer
NWSL
North American League
UEFA Europa League


In [6]:
# selected_leagues = ["La Liga", "Serie A", "Premier League", "1. Bundesliga", "Ligue 1", "Champions League", "UEFA Europa League"]
women_leagues = ["FA Women's Super League", "Women's World Cup", "UEFA Women's Euro"]
# selected_leagues_mask = df_competitions['competition_name'].isin(selected_leagues)
selected_leagues_mask = df_competitions['competition_name'].isin(women_leagues)
df_competitions = df_competitions[selected_leagues_mask]

In [7]:
df_competitions.value_counts(subset=['competition_name'])

competition_name       
FA Women's Super League    3
UEFA Women's Euro          2
Women's World Cup          2
Name: count, dtype: int64

In [8]:
df_competitions

,season_id,competition_id,competition_name,country_name,competition_gender,season_name
25,90,37,FA Women's Super League,England,female,2020/2021
26,42,37,FA Women's Super League,England,female,2019/2020
27,4,37,FA Women's Super League,England,female,2018/2019
71,315,53,UEFA Women's Euro,Europe,female,2025
72,106,53,UEFA Women's Euro,Europe,female,2022
73,107,72,Women's World Cup,International,female,2023
74,30,72,Women's World Cup,International,female,2019


## Select league/season

Edit `selected_name_pairs` to include the league(s) and season(s) you want.

- The `competition_name` and `season_name` must match exactly the values shown in the table above.
- If there are duplicates, use `selected_id_pairs` instead (competition_id, season_id).

In [9]:
# Select league/season pairs to process
SAVE_ALL_AVAILABLE = True  # True => process every available competition+season in df_selected_competitions

# OPTION A: select by names (used only when SAVE_ALL_AVAILABLE = False)
selected_name_pairs = [
    # ('Serie A', '2015/2016'),
]

# OPTION B: select by ids (used only when SAVE_ALL_AVAILABLE = False)
selected_id_pairs = [
    # (11, 27),
]

if SAVE_ALL_AVAILABLE:
    selected = [
        (int(row.competition_id), int(row.season_id))
        for row in df_competitions[['competition_id', 'season_id']].drop_duplicates().itertuples(index=False)
    ]
elif selected_name_pairs and selected_id_pairs:
    raise ValueError('Use either selected_name_pairs OR selected_id_pairs (not both).')
elif selected_name_pairs:
    selected = [resolve_competition_season_ids(df_competitions, c, s) for (c, s) in selected_name_pairs]
elif selected_id_pairs:
    selected = [(int(c), int(s)) for (c, s) in selected_id_pairs]
else:
    selected = []

print('Total selected (competition_id, season_id):', len(selected))
print('Preview:', selected[:10])

Total selected (competition_id, season_id): 7
Preview: [(37, 90), (37, 42), (37, 4), (53, 315), (53, 106), (72, 107), (72, 30)]


In [ ]:
# Build and save one dataset per selected competition/season
if not selected:
    raise ValueError('No league/season selected. Configure SAVE_ALL_AVAILABLE or selection lists above.')

outputs = []
failed = []

for competition_id, season_id in selected:
    try:
        fpath, lpath, meta = build_and_save_vaep_for_competition_season(
            loader=SBL,
            competitions_df=df_competitions,
            output_dir=output_dir,
            competition_id=competition_id,
            season_id=season_id,
        )
        outputs.append((fpath, lpath))
        if meta.get('games_failed'):
            print(f'  Warning: {meta["games_failed"]} games failed')
    except Exception as e:
        failed.append((competition_id, season_id, str(e)))
        print(f'Skipped (cid={competition_id}, sid={season_id}): {e}')

print('\nDone.')
print('Saved datasets:', len(outputs))
print('Failed datasets:', len(failed))

if outputs:
    print('\nSaved files:')
    for fpath, lpath in outputs:
        print('-', fpath)
        print('-', lpath)

if failed:
    print('\nFirst failed items:')
    display(pd.DataFrame(failed, columns=['competition_id', 'season_id', 'error']).head(10))


=== FA Women's Super League | 2020/2021 (cid=37, sid=90) ===
-> ../data/spadl_data_women_leagues/features_fa_women_s_super_league_2020_2021.h5
-> ../data/spadl_data_women_leagues/labels_fa_women_s_super_league_2020_2021.h5
converted 50/131 games
converted 100/131 games
Saved features: (253175, 38)
Saved labels:   (253175, 4)

=== FA Women's Super League | 2019/2020 (cid=37, sid=42) ===
-> ../data/spadl_data_women_leagues/features_fa_women_s_super_league_2019_2020.h5
-> ../data/spadl_data_women_leagues/labels_fa_women_s_super_league_2019_2020.h5
converted 50/87 games
Saved features: (166601, 38)
Saved labels:   (166601, 4)

=== FA Women's Super League | 2018/2019 (cid=37, sid=4) ===
-> ../data/spadl_data_women_leagues/features_fa_women_s_super_league_2018_2019.h5
-> ../data/spadl_data_women_leagues/labels_fa_women_s_super_league_2018_2019.h5
converted 50/108 games
converted 100/108 games
Saved features: (208074, 38)
Saved labels:   (208074, 4)

=== UEFA Women's Euro | 2025 (cid=53, sid

In [11]:
trainval_leagues = ["FA Women's Super League"]
test_leagues = ["Women's World Cup", "UEFA Women's Euro"]

selected_trainval = df_competitions[df_competitions.competition_name.isin(trainval_leagues)].value_counts(subset=['competition_id', 'season_id']).index.tolist()
selected_test = df_competitions[df_competitions.competition_name.isin(test_leagues)].value_counts(subset=['competition_id', 'season_id']).index.tolist()

In [ ]:
# Train a RandomForest on ALL selected data (no split)
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Collect (features, labels) from saved files for each selected comp/season
df_all_train = []
for competition_id, season_id in selected_trainval:
    row = df_competitions[(df_competitions.competition_id == competition_id) & (df_competitions.season_id == season_id)]
    if len(row) != 1:
        raise ValueError(f'Cannot uniquely resolve competition_id={competition_id}, season_id={season_id}')
    comp_name = str(row.iloc[0].competition_name)
    season_name = str(row.iloc[0].season_name)
    league_slug = slug(comp_name)
    season_slug = slug(season_name)
    features_path = os.path.join(output_dir, f'features_{league_slug}_{season_slug}.h5')
    labels_path = os.path.join(output_dir, f'labels_{league_slug}_{season_slug}.h5')
    if not (os.path.exists(features_path) and os.path.exists(labels_path)):
        raise FileNotFoundError(
            f'Missing files for {comp_name} {season_name}. Expected:\n- {features_path}\n- {labels_path}\nRun the dataset-generation cell first.'
        )

    with pd.HDFStore(features_path, mode='r') as store:
        df_features = store['features'].copy()
    with pd.HDFStore(labels_path, mode='r') as store:
        df_labels = store['labels'].copy()

    df = df_features.merge(df_labels, on=['game_id', 'action_id'], how='inner')
    df['competition_id'] = competition_id
    df['season_id'] = season_id
    df_all_train.append(df)
    print(f'Loaded + joined {comp_name} {season_name}:', df.shape)

df_all_train = pd.concat(df_all_train, ignore_index=True)
print('\nTotal joined dataset:', df_all_train.shape)

# Build X/y (no split)
drop_cols = {'game_id', 'action_id', 'scores', 'concedes', 'competition_id', 'season_id'}
feature_cols = [c for c in df_all_train.columns if c not in drop_cols]
X = df_all_train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_scores = df_all_train['scores'].astype(int)
y_concedes = df_all_train['concedes'].astype(int)

print('X:', X.shape, 'y_scores:', y_scores.shape, 'y_concedes:', y_concedes.shape)
for c in feature_cols:
    print(f'- {c}: {X[c].dtype}, min={X[c].min()}, max={X[c].max()}')

print('Score and concede rates:')
print('Score rate:', y_scores.sum() / y_scores.shape[0])
print('Concede rate:', y_concedes.sum() / y_concedes.shape[0])

# RandomForest (baseline)
rf_scores = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample',
)

rf_scores.fit(X, y_scores)

print('Trained RandomForest models on full dataset (no split).')

Loaded + joined FA Women's Super League 2018/2019: (208074, 42)
Loaded + joined FA Women's Super League 2019/2020: (166601, 42)
Loaded + joined FA Women's Super League 2020/2021: (253175, 42)

Total joined dataset: (627850, 42)
X: (627850, 36) y_scores: (627850,) y_concedes: (627850,)
- result_fail_a0: bool, min=False, max=True
- result_success_a0: bool, min=False, max=True
- result_offside_a0: bool, min=False, max=True
- result_owngoal_a0: bool, min=False, max=True
- result_yellow_card_a0: bool, min=False, max=True
- result_red_card_a0: bool, min=False, max=True
- result_fail_a1: bool, min=False, max=True
- result_success_a1: bool, min=False, max=True
- result_offside_a1: bool, min=False, max=True
- result_owngoal_a1: bool, min=False, max=True
- result_yellow_card_a1: bool, min=False, max=True
- result_red_card_a1: bool, min=False, max=True
- result_fail_a2: bool, min=False, max=True
- result_success_a2: bool, min=False, max=True
- result_offside_a2: bool, min=False, max=True
- result

In [ ]:
# Build test set datasets
print('Selected (competition_id, season_id):', selected_test)

if not selected_test:
    raise ValueError('No league/season selected. Edit selected_name_pairs or selected_id_pairs above.')

outputs = []
for competition_id, season_id in selected_test:
    fpath, lpath, meta = build_and_save_vaep_for_competition_season(
        loader=SBL,
        competitions_df=df_competitions,
        output_dir=output_dir,
        competition_id=competition_id,
        season_id=season_id,
    )
    outputs.append((fpath, lpath))

print('\nDone. Outputs:')
for fpath, lpath in outputs:
    print('-', fpath)
    print('-', lpath)

Selected (competition_id, season_id): [(53, 106), (53, 315), (72, 30), (72, 107)]

=== UEFA Women's Euro | 2022 (cid=53, sid=106) ===
-> ../data/spadl_data_women_leagues/features_uefa_women_s_euro_2022.h5
-> ../data/spadl_data_women_leagues/labels_uefa_women_s_euro_2022.h5
Saved features: (61660, 38)
Saved labels:   (61660, 4)

=== UEFA Women's Euro | 2025 (cid=53, sid=315) ===
-> ../data/spadl_data_women_leagues/features_uefa_women_s_euro_2025.h5
-> ../data/spadl_data_women_leagues/labels_uefa_women_s_euro_2025.h5
Saved features: (60370, 38)
Saved labels:   (60370, 4)

=== Women's World Cup | 2019 (cid=72, sid=30) ===
-> ../data/spadl_data_women_leagues/features_women_s_world_cup_2019.h5
-> ../data/spadl_data_women_leagues/labels_women_s_world_cup_2019.h5
converted 50/52 games
Saved features: (101575, 38)
Saved labels:   (101575, 4)

=== Women's World Cup | 2023 (cid=72, sid=107) ===
-> ../data/spadl_data_women_leagues/features_women_s_world_cup_2023.h5
-> ../data/spadl_data_women_lea

In [ ]:
# Collect (features, labels) from saved files for each selected comp/season
import numpy as np

df_all_test = []
for competition_id, season_id in selected_test:
    row = df_competitions[(df_competitions.competition_id == competition_id) & (df_competitions.season_id == season_id)]
    if len(row) != 1:
        raise ValueError(f'Cannot uniquely resolve competition_id={competition_id}, season_id={season_id}')
    comp_name = str(row.iloc[0].competition_name)
    season_name = str(row.iloc[0].season_name)
    league_slug = slug(comp_name)
    season_slug = slug(season_name)
    features_path = os.path.join(output_dir, f'features_{league_slug}_{season_slug}.h5')
    labels_path = os.path.join(output_dir, f'labels_{league_slug}_{season_slug}.h5')
    if not (os.path.exists(features_path) and os.path.exists(labels_path)):
        raise FileNotFoundError(
            f'Missing files for {comp_name} {season_name}. Expected:\n- {features_path}\n- {labels_path}\nRun the dataset-generation cell first.'
        )

    with pd.HDFStore(features_path, mode='r') as store:
        df_features = store['features'].copy()
    with pd.HDFStore(labels_path, mode='r') as store:
        df_labels = store['labels'].copy()

    df = df_features.merge(df_labels, on=['game_id', 'action_id'], how='inner')
    df['competition_id'] = competition_id
    df['season_id'] = season_id
    df_all_test.append(df)
    print(f'Loaded + joined {comp_name} {season_name}:', df.shape)

df_test = pd.concat(df_all_test, ignore_index=True)
print('\nTotal joined dataset:', df_test.shape)

# Build X/y (no split)
drop_cols = {'game_id', 'action_id', 'scores', 'concedes', 'competition_id', 'season_id'}
feature_cols = [c for c in df_test.columns if c not in drop_cols]
X = df_test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_scores = df_test['scores'].astype(int)
y_concedes = df_test['concedes'].astype(int)

print('X:', X.shape, 'y_scores:', y_scores.shape, 'y_concedes:', y_concedes.shape)

Loaded + joined UEFA Women's Euro 2022: (61660, 42)
Loaded + joined UEFA Women's Euro 2025: (60370, 42)
Loaded + joined Women's World Cup 2019: (101575, 42)
Loaded + joined Women's World Cup 2023: (125680, 42)

Total joined dataset: (349285, 42)
X: (349285, 36) y_scores: (349285,) y_concedes: (349285,)


In [ ]:
# Test the trained RandomForest on this test dataset

# Align feature columns to what the model was trained on (handles missing/extra columns)
if hasattr(rf_scores, 'feature_names_in_'):
    train_cols = list(rf_scores.feature_names_in_)
    X_test_aligned = X.reindex(columns=train_cols, fill_value=0)
else:
    X_test_aligned = X.copy()

# Predict probabilities using library helper
p_scores = get_positive_class_scores(rf_scores, X_test_aligned)

# Evaluate using library helper
threshold = 0.5
metrics = evaluate_binary(p_scores, y_scores, threshold=threshold)
print('[scores] Evaluation metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.6f}')

# Preview a few predictions alongside labels
yhat_scores = (p_scores >= threshold).astype(int)
df_pred = df_test[['game_id', 'action_id', 'scores', 'concedes']].copy()
df_pred['p_scores'] = p_scores
df_pred['yhat_scores'] = yhat_scores
display(df_pred.head(10))


[scores]
ROC AUC:   0.7649550500407948
PR AUC:    0.19882141849821316
Log loss:  0.10321332291032075
Brier:     0.010481188785662138

[scores:] (threshold=0.5)
Accuracy:  0.9890118384700173
Precision: 0.9784615384615385
Recall:    0.07664497469269703
F1:        0.14215467143495752


,game_id,action_id,scores,concedes,p_scores,yhat_scores
0,3835331,0,False,False,0.000,0
1,3835331,1,False,False,0.000,0
2,3835331,2,False,False,0.000,0
3,3835331,3,False,False,0.005,0
4,3835331,4,False,False,0.005,0
5,3835331,5,False,False,0.020,0
6,3835331,6,False,False,0.000,0
7,3835331,7,False,False,0.000,0
8,3835331,8,False,False,0.000,0
9,3835331,9,False,False,0.000,0


In [ ]:
from sklearn.metrics import confusion_matrix

test_cm = confusion_matrix(y_scores, yhat_scores)
df_cm = pd.DataFrame(
    test_cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"],
)
tn, fp, fn, tp = test_cm.ravel()
print("\n=== TEST ===")
print(df_cm)
print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")


=== TEST ===
Predicted        Predicted Negative  Predicted Positive
Actual                                                 
Actual Negative              345129                   7
Actual Positive                3831                 318
TP: 318, TN: 345129, FP: 7, FN: 3831


Predicted,Predicted Negative,Predicted Positive
Actual,,
Actual Negative,345129,7
Actual Positive,3831,318
